Linear Regression
This notebook implements Linear Regression two ways: from scratch using gradient descent, and using scikit-learn. We fit a line y = w*x + b to noisy synthetic data and evaluate the model using R-squared and Mean Squared Error.

Cell 1: Imports and Setup

In [ ]:
# Manasa Basavaraja
# Linear Regression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Fix the random seed so results are reproducible across runs
np.random.seed(42)

Cell 2: Dataset Generation

In [ ]:
# Build a synthetic linear dataset:  y = 3*x + 7  with Gaussian noise added.
# This mirrors the style used in PolynomialRegression.ipynb.
n_samples = 100
x = np.random.uniform(-5, 5, n_samples)
true_slope = 3.0
true_intercept = 7.0
noise = np.random.normal(0, 2.0, n_samples)
y = true_slope * x + true_intercept + noise

# Wrap the values in a dataframe for quick inspection.
dataframe = pd.DataFrame({'x': x, 'y': y})
dataframe.head()

Cell 3: Initial Scatter Plot

In [ ]:
# Visualize the raw (x, y) data points before fitting any model.
plt.scatter(dataframe['x'], dataframe['y'], color='#008B8B', label='Data points')
plt.title('Scatter plot of the (x, y) coordinates')
plt.xlabel('x-axis')
plt.ylabel('y-axis')
plt.legend()
plt.show()

Cell 4: Train / Test Split

In [ ]:
# Reshape x into a column vector since sklearn expects 2D feature arrays.
X = dataframe['x'].values.reshape(-1, 1)
y_vals = dataframe['y'].values

X_train, X_test, y_train, y_test = train_test_split(X, y_vals, test_size=0.2, random_state=42)
print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples : {X_test.shape[0]}')

Cell 5: Linear Regression from Scratch (Gradient Descent)

In [ ]:
def fit_linear_regression(X, y, learning_rate=0.01, epochs=1000):
    """
    Fit a simple linear regression model y = w*x + b using batch gradient descent.
    The cost minimized is the Mean Squared Error.

    Returns the learned weight, bias, and the loss recorded at each epoch.
    """
    # Flatten to 1D for our scratch implementation
    x_flat = X.ravel()
    m = len(x_flat)

    # Initialize parameters to zero
    w = 0.0
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        # Forward pass: prediction for every sample
        y_pred = w * x_flat + b
        error = y_pred - y

        # Gradients of MSE with respect to w and b
        dw = (2 / m) * np.dot(error, x_flat)
        db = (2 / m) * np.sum(error)

        # Update step
        w -= learning_rate * dw
        b -= learning_rate * db

        # Track loss so we can plot the convergence curve later
        loss = np.mean(error ** 2)
        loss_history.append(loss)

    return w, b, loss_history

w_scratch, b_scratch, loss_history = fit_linear_regression(X_train, y_train, learning_rate=0.01, epochs=1000)
print(f'Learned slope (w)    : {w_scratch:.4f}')
print(f'Learned intercept (b): {b_scratch:.4f}')
print(f'True slope / intercept were {true_slope} and {true_intercept}.')

Cell 6: Loss Curve

In [ ]:
# Plot how the MSE drops as gradient descent runs.
plt.plot(loss_history, color='#A52A2A')
plt.title('Gradient Descent: Loss vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.show()

Cell 7: Linear Regression using scikit-learn

In [ ]:
# Same fit using sklearn so we can compare against the scratch implementation.
sk_model = LinearRegression()
sk_model.fit(X_train, y_train)

print(f'sklearn slope (w)    : {sk_model.coef_[0]:.4f}')
print(f'sklearn intercept (b): {sk_model.intercept_:.4f}')

Cell 8: Predictions and Comparison Plot

In [ ]:
# Generate predictions from both models for visualization.
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_scratch_line = w_scratch * x_line.ravel() + b_scratch
y_sklearn_line = sk_model.predict(x_line)

plt.scatter(X_train, y_train, color='#008B8B', alpha=0.6, label='Train')
plt.scatter(X_test, y_test, color='#A52A2A', alpha=0.8, label='Test')
plt.plot(x_line, y_scratch_line, color='blue', linewidth=2, label='From scratch')
plt.plot(x_line, y_sklearn_line, color='orange', linewidth=2, linestyle='--', label='sklearn')
plt.title('Linear Regression: Scratch vs. sklearn')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

Cell 9: Evaluation

In [ ]:
# Compare model accuracy on the held-out test set.
y_pred_scratch = w_scratch * X_test.ravel() + b_scratch
y_pred_sklearn = sk_model.predict(X_test)

print('Scratch implementation:')
print(f'  MSE: {mean_squared_error(y_test, y_pred_scratch):.4f}')
print(f'  R^2: {r2_score(y_test, y_pred_scratch):.4f}')

print('\nsklearn implementation:')
print(f'  MSE: {mean_squared_error(y_test, y_pred_sklearn):.4f}')
print(f'  R^2: {r2_score(y_test, y_pred_sklearn):.4f}')

# Note: Both implementations should converge to nearly identical parameters
# and produce very similar MSE / R^2 values. Any small differences come from
# gradient descent stopping before reaching the exact closed-form optimum.